# Buckley Steerer Degauss — Interactive GUI Walkthrough

Drive the `rotating_coil_analyzer` **ipywidgets GUI** on the *StandardDegauss*
Buckley-steerer dataset, with the exact control settings for every tab.

The ingest layer now reads the FFMM degauss format **natively** — both the
`<magnet>_<timestamp>_Parameters.txt` filename and the H/V plateau naming
`..._Run_<step>IH_<ih>IV_<iv>_<seg>_raw_measurement_data.txt`. So you just point
the **Catalog** tab at the original measurement folder; no renaming or staging
copy is needed.

> For exact FFMM golden parity and batch processing of all 74 runs, use
> `parity_validation.ipynb` in this folder. This notebook is for **interactive
> exploration**.

## Important notes before you start

**1. Skip the Plateau Detection tab.** This is **plateau (staircase) data**:
each degauss step is a separate file, and the reader tags every turn with a
`plateau_id`. The Harmonics, Coil Calibration, Harmonic Merge and Physics tabs
read `plateau_id` directly, so the Plateau Detection tab (meant for streaming
supercycles) is not needed.

**2. The displayed current is a reference, not the magnet drive.** The magnet
ran at $\le 10$ A, but column 4 of these files is a near-constant reference
(~1000 in its own raw units) that reads the same at $+10$ A, $-10$ A and $0$ A.
The **true** per-run current is the commanded set current, which the reader
parses from the filename into the **`plateau_I_hint`** column (signed, e.g.
$+10, -10, +6.667, \ldots$ A). So:

- The GUI's current readout / transfer-function / hysteresis x-axis show the
  ~1000 reference and are **not physical** for this dataset.
- For current-referenced results use `plateau_I_hint` or the
  `parity_validation.ipynb` (which uses the commanded current).
- We deliberately do **not** overwrite the data — `plateau_I_hint` already
  carries the truth, with no edits to the measured file.

**3. The harmonics do not use the current at all.** The only stage that would is
the `dit` di/dt correction, and it can never even activate here (it needs a ramp:
$|\mathrm{d}I/\mathrm{d}t|>0.1$ A/s **and** $|\overline{I}|>10$ A, but every run
is a fixed-current plateau). So $B_n/A_n$ — and the machine-precision FFMM parity
— are independent of the current column.

**4. 20 turns per run, current valid for the first 10.** Each file holds 20
turns; the current column is filled only for the first 10 (turns 10–19 are
NaN-current). The flux is valid for all 20, so the GUI computes harmonics for
every turn; FFMM analysed the first 10 (see the parity notebook).

## What each tab does for this dataset

| Tab | Use here |
|-----|----------|
| **0 Catalog** | Browse to the measurement folder, Discover, Load segment **Main** (74 degauss steps as one staircase, one `plateau_id` per step). Preview waveforms. |
| **1 Plateau Detection** | **Skip** — data is already segmented per run. |
| **2 Harmonics** | Preview/apply the QC cuts, compute FFT harmonics per turn, view amplitude & normal/skew per plateau. |
| **3 Coil Calibration** | Load `Kn_values_Seg_Main.txt`; declare compensation scheme `A-C`. |
| **4 Harmonic Merge** | Apply Kn with the FFMM settings and export a traceable CSV. |
| **5 Raw Signal Plots** | Explore `df_abs/df_cmp` vs time (decimation only). |
| **6 Physics Plots** | Per-plateau harmonic statistics. (B-vs-I axis uses the reference column — not physical here; see note 2.) |
| **7 Comparison** | Overlay two exported CSVs (e.g. two degauss sub-runs). |

## What the tab settings actually do (pipeline primer)

The ticks below are not arbitrary — they switch stages of a fixed analysis
chain. Full conceptual writeup (with formulas):
[`../../pipeline_reference.md`](../../pipeline_reference.md). In brief:

```
raw increments df  →  [dit] di/dt  →  [dri] integrate-to-flux + drift
                   →  FFT  →  [Kn] calibrate  →  [rot] phase-align
                   →  [cel] centre  →  [fed] feed-down  →  merge abs/cmp  →  [nor] normalise
```

- **Integrate differential signal to flux** — the FDI outputs flux *increments*
  $\mathrm{d}\Phi$, not flux. Summing them (`cumsum`) rebuilds $\Phi(\theta)$,
  whose FFT gives the harmonics. (A cumulative *sum*, not trapezoid, because the
  signal is already differential.) Mandatory.
- **Drift correction (`dri`)** — removes the integrator's DC baseline so the
  flux closes over one turn. *Legacy* = uniform $\Delta t$ (FFMM parity);
  *Weighted* = $\Delta t$-weighted (uneven sample timing).
- **di/dt (`dit`)** — ramps only; no-op on plateaus → OFF here.
- **Kn** — converts flux harmonics to physical multipoles (per channel).
- **rot** — aligns the phase using the *calibrated* main harmonic; after Kn.
- **cel/fed** — locate the magnetic centre and re-expand about it; OFF here.
- **merge** — main field from Abs, small orders from Cmp.
- **normalise (`nor`)** — ratio to the main field, applied **last and after
  merge**; OFF here so output stays in Tesla (matching the golden).

In [ ]:
from pathlib import Path
from rotating_coil_analyzer.ingest.discovery import find_parameters_txt


def _find_repo_root(start: Path) -> Path:
    p = start.resolve()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    return start.resolve()

REPO = _find_repo_root(Path.cwd())

# The original measurement folder — point the Catalog tab here directly.
# (Switch to ..._144731 / ..._150306 or SpiralDegauss to explore another run.)
DATA = (REPO / "golden_standards" / "degaussing_test" / "StandardDegauss"
        / "20260623_165352_Carbonara_test_Buckley-steerer")

print("Browse the Catalog tab to:\n  ", DATA)
print("Parameters file the discovery will use:", find_parameters_txt(DATA).name)
print("Kn file to load in Coil Calibration:   ", (DATA / "Kn_values_Seg_Main.txt").name)

## Step-by-step in the GUI (exact settings)

Run the launch cell, then go **left to right**. **Bold** = the control to set;
leave anything not mentioned at its default.

### Tab 0 — Catalog
1. **Browse** -> select the `DATA` folder printed above.
2. **Discover** -> it lists aperture 1, segment **Main**.
3. Select **Main** -> **Load**. All 74 degauss steps load as one staircase
   (one `plateau_id` per step; 20 turns each).
4. The preview shows the first turns of `df_abs / df_cmp` — confirm clean
   sinusoids. (The current trace is the ~1000 reference — see note 2 above.)

### Tab 1 — Plateau Detection
**Skip it** (data is already split per run).

### Tab 2 — Harmonics *(raw spectrum — no calibration needed)*
This tab computes the per-turn FFT and shows amplitude / normal-skew **without**
applying Kn, so it is independent of Coil Calibration. Tick boxes — set:
- **Main field order = 1**
- **Maximum harmonic order = 15**
- **Integrate differential signal to flux** = **ON** (channels are flux
  increments; required).
- **Apply drift correction** = **ON**, **Drift mode = Legacy (C++)** (matches
  the FFMM golden `dri`).
- **Apply di/dt correction** = **OFF**.
- **Require valid time** = **ON**.

Then **Preview data-quality cuts** -> **Apply cuts and compute harmonics (FFT)**,
and step through plateaus with the **Plateau selection** dropdown. Tick **Hide
main field order** to see the small higher-order terms.

### Tab 3 — Coil Calibration *(needed only for the Harmonic Merge below)*
1. **Load Kn from TXT** -> **Browse** to `Kn_values_Seg_Main.txt` in the `DATA`
   folder -> **Load**.
2. **Compensation scheme = `A-C`** — the active kn is `Kn_R45_N1_A_AC.txt`
   (absolute coil A, compensated A-C, dipole bucking). Metadata only.

### Tab 4 — Harmonic Merge
1. **Magnet order = 1**, **Rref = 0.04 m**.
2. Options: **dri = ON**, **rot = ON**; **cel = OFF**, **fed = OFF**,
   **nor = OFF** (reproduces the FFMM golden; output in Tesla).
3. **Merge preset = "main from Abs, others from Cmp"**.
4. **Legacy rotate excludes last = ON** for exact FFMM parity; leave **OFF** for
   the physically-correct convention (rotate all harmonics, differs only at the
   last order $n=15$).
5. **Apply merge**, then **Export** the CSV.

### Tab 6 — Physics Plots
- **N last turns** (e.g. 10) -> **Compute summary** for per-plateau mean/std of
  $B_1$ and the harmonics.
- The hysteresis / transfer-function panels plot against the recorded current,
  which here is the ~1000 reference — **not physical** for this dataset (note 2).
  For $B_1$ vs the true current, use `plateau_I_hint` or `parity_validation.ipynb`.

### Tab 7 — Comparison
- Export a second sub-run (point `DATA` at `..._144731` / `..._150306`, re-run,
  export) and load the two exported CSVs here to compare degauss residuals.

In [ ]:
%matplotlib widget
from rotating_coil_analyzer.gui.app import build_gui

gui = build_gui()
gui

---
*Interactive launcher — the cell above renders the live GUI and is intentionally
left unexecuted in the committed notebook. For reproducible, scripted analysis
and exact FFMM parity, use `parity_validation.ipynb`.*